# Staples Marketplace PoC – category-level recommendations (v2, six retailers)

Runs the whole pipeline end to end:

1. **Method V** – path-aware embeddings + Qdrant vector DB (`method_v_vector.py`)
2. **Method G** – category knowledge graph, similarity flooding, PageRank (`method_g_graph.py`)
3. **Framework** – ensemble, ENTER/DEEPEN units, Track C scores, six zones, all figures and the workbook (`run_framework.py`)

**Before you start:** put these files in one Google Drive folder (or upload them in the next cell):
`poc_common.py, method_v_vector.py, method_g_graph.py, run_framework.py, framework_outputs.py, label_sample.py, gold_matches_v2.csv`
and the six trees: `Staples_Navigation_Tree*.xlsx, OfficeDepot_Navigation_Tree.xlsx, WestElm_Navigation_Tree.xlsx, Wayfair_Navigation_Tree*.xlsx, Amazon_Navigation_Tree*.xlsx, Walmart_Navigation_Tree*.xlsx`.

Runtime on a standard Colab CPU: about 6-8 minutes.

## 1. Install

In [ ]:
!pip install -q pandas numpy scipy scikit-learn openpyxl matplotlib networkx qdrant-client spacy adjustText
!python -m spacy download en_core_web_lg -q
# Optional encoders for the bake-off (set CONFIG["encoder"]="auto" in poc_common.py to let the data pick):
# !pip install -q sentence-transformers      # BAAI/bge-small-en-v1.5 (needs HuggingFace access - fine in Colab)
# !pip install -q wordllama                  # small static embeddings shipped in the wheel

## 2. Point to your files
Either mount Drive and set `WORK` to the folder, or leave `WORK='/content'` and upload.

In [ ]:
import os
WORK = '/content'                      # e.g. '/content/drive/MyDrive/staples_poc'
# from google.colab import drive; drive.mount('/content/drive')
# from google.colab import files; files.upload()     # uploads land in /content
os.chdir(WORK)
os.environ['POC_DATA_DIR'] = WORK
os.environ['POC_OUT_DIR'] = os.path.join(WORK, 'outputs')
print(sorted(f for f in os.listdir(WORK) if f.endswith(('.py', '.xlsx', '.csv'))))

## 3. Method V – vectors + Qdrant (≈4 min incl. encoder bake-off)

In [ ]:
!python method_v_vector.py

## 4. Method G – knowledge graph (≈30 s)

In [ ]:
!python method_g_graph.py

## 5. Framework – units, scores, six zones, figures, workbook (≈1 min)

In [ ]:
!python run_framework.py

## 6. Key results

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
S = json.load(open('outputs/final/summary.json'))
print('Aisles per zone:', S['family_counts'])
print('Sub-categories (2+ competitors) per zone:', S['recommendable_units_by_zone'])
display(pd.DataFrame(S['families']['CURATE'])[['family_display','examples','competitors','AAS','CRS','O_sum','best_evidence']])
for f in ['fig06_matrix_aisles.png', 'fig08_zone_curate.png', 'fig14_themes.png', 'fig17_graph_dock.png']:
    display(Image(f'outputs/final/figures/{f}', width=1000))

## 7. Download the outputs

In [ ]:
!cd outputs && zip -qr ../staples_poc_outputs.zip final G/neo4j G/node_scores_G.csv V/node_scores_V.csv V/encoder_bakeoff.csv V/context_weight_grid.csv
# from google.colab import files; files.download('staples_poc_outputs.zip')

## Adding a competitor
1. Add one adapter to `RETAILERS` in `poc_common.py` (file pattern, level columns, count column + `count_mode`, role, scope rules).
2. Run cell 3 – the new competitor runs on pooled calibration (flagged provisional).
3. `!python label_sample.py NewCompetitor` → label the 120 rows in `outputs/labelling/labelling_sample_NewCompetitor.csv`, append to `gold_matches_v2.csv`, re-run cells 3–5.